# Exercise XAI3: Partial Dependence Plots and Model Interpretability

**Authors:** Lucía Cardona Ramón, Lucía Guillén López, Clara Pascual Olloqui

This notebook provides the R code and explanations for the requested interpretability analysis. The objective is to train Random Forest regression models and explain their behaviour using Partial Dependence Plots (PDPs).

## 0. Running the notebook in Google Colab

Use an **R runtime** in Colab. If the runtime selector does not show R, create the notebook from a Colab R template or open this `.ipynb` with an R kernel.

Before running the code, upload the `DATA` folder to the working directory. The expected files are:

- `DATA/day.csv`
- `DATA/hour.csv`
- `DATA/kc_house_data.csv`

The code uses relative paths and creates a `figures` folder automatically. The original datasets are only read, never modified.

In [4]:
# Check that the DATA folder is available in the current working directory.
getwd()
list.files()
list.files("DATA")

[1] "/content"

[1] "figures"     "sample_data"

character(0)

## 1. Libraries and reproducibility

The analysis uses `randomForest` for modelling, `pdp` for Partial Dependence Plots, and `ggplot2` for visualisation. The seed is fixed to make the train/test splits and samples reproducible.

In [5]:
set.seed(123)

required_packages <- c(
  "randomForest", "pdp", "ggplot2", "dplyr", "patchwork", "viridis"
)

missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_packages) > 0) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}

library(randomForest)
library(pdp)
library(ggplot2)
library(dplyr)
library(patchwork)
library(viridis)

dir.create("figures", showWarnings = FALSE, recursive = TRUE)
theme_set(theme_minimal(base_size = 12))

## 2. Helper functions

These functions keep the notebook concise. The evaluation function returns RMSE, MAE and R². The PDP helper computes a one-dimensional PDP and adds rug marks to show where observations are located.

In [8]:
save_plot <- function(plot, filename, width = 7, height = 5) {
  ggsave(
    filename = file.path("figures", filename),
    plot = plot,
    width = width,
    height = height,
    dpi = 300,
    bg = "white"
  )
}

train_test_split <- function(data, train_prop = 0.8) {
  train_index <- sample(seq_len(nrow(data)), size = floor(train_prop * nrow(data)))
  list(train = data[train_index, , drop = FALSE], test = data[-train_index, , drop = FALSE])
}

regression_metrics <- function(actual, predicted) {
  data.frame(
    RMSE = sqrt(mean((actual - predicted)^2)),
    MAE = mean(abs(actual - predicted)),
    R2 = 1 - sum((actual - predicted)^2) / sum((actual - mean(actual))^2)
  )
}

make_1d_pdp <- function(model, data, feature, target_label, x_label = feature,
                        grid.resolution = 50) {
  pd <- partial(
    object = model,
    pred.var = feature,
    train = data,
    grid.resolution = grid.resolution
  )

  ggplot(pd, aes(x = .data[[feature]], y = yhat)) +
    geom_line(color = "#1f78b4", linewidth = 1.1) +
    geom_rug(data = data, aes(x = .data[[feature]], y = NULL),
             inherit.aes = FALSE, sides = "b", alpha = 0.16) +
    labs(x = x_label, y = paste("Partial dependence of", target_label))
}

## 3. Dataset choice

The bike sharing data include both daily and hourly files. The exercise asks for PDPs involving `instant`, `temp`, `hum`, `windspeed` and the target `cnt`. The daily file is used because it directly represents daily rental counts and makes `instant` interpretable as days since the beginning of 2011. The hourly file is loaded only to document the decision; using it would add hourly behaviour that is outside the requested interpretation.

In [ ]:
# If the notebook is inside CODE/, the project root is one level above
if (dir.exists("../DATA")) {
  project_root <- normalizePath("..")
} else if (dir.exists("DATA")) {
  project_root <- normalizePath(".")
} else {
  stop("DATA folder not found. Make sure DATA is in the project root or upload it to the notebook environment.")
}

data_dir <- file.path(project_root, "DATA")

# Check available files
list.files(data_dir)

ERROR: Error: DATA folder not found. Make sure DATA is in the project root or upload it to the notebook environment.


In [9]:
bike_day_raw <- read.csv(file.path("DATA", "day.csv"))
bike_hour_raw <- read.csv(file.path("DATA", "hour.csv"))
house_raw <- read.csv(file.path("DATA", "kc_house_data.csv"))

data.frame(
  Dataset = c("day.csv", "hour.csv", "kc_house_data.csv"),
  Rows = c(nrow(bike_day_raw), nrow(bike_hour_raw), nrow(house_raw)),
  Columns = c(ncol(bike_day_raw), ncol(bike_hour_raw), ncol(house_raw))
)

Warning message in file(file, "rt"):
“cannot open file 'DATA/day.csv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


## 4. Bike rental prediction: data preparation and model training

The Random Forest model predicts daily bike rental count `cnt` using the requested variables. An 80/20 train/test split is used to check predictive quality before interpreting the model.

In [ ]:
bike_data <- bike_day_raw %>%
  select(cnt, instant, temp, hum, windspeed) %>%
  mutate(across(everything(), as.numeric)) %>%
  na.omit()

bike_split <- train_test_split(bike_data)
bike_train <- bike_split$train
bike_test <- bike_split$test

bike_rf <- randomForest(
  cnt ~ instant + temp + hum + windspeed,
  data = bike_train,
  ntree = 300,
  importance = TRUE
)

bike_predictions <- predict(bike_rf, newdata = bike_test)
bike_metrics <- regression_metrics(bike_test$cnt, bike_predictions) %>%
  mutate(Dataset = "Bike rentals", .before = RMSE)

bike_rf
bike_metrics

## 5. Bike rental prediction: PDP for days since 2011

This PDP explains how the model prediction changes across the observed timeline.

In [ ]:
bike_pdp_instant <- make_1d_pdp(
  bike_rf, bike_train, "instant", "bike count", "Days since 2011"
)
save_plot(bike_pdp_instant, "bike_pdp_instant.png")
bike_pdp_instant

**Interpretation.** The effect is non-linear. The model prediction generally increases during much of the observed period, rises sharply around the middle of the timeline, and falls near the end. This should be interpreted as a time-related model pattern, not as causal evidence that time itself creates demand.

## 6. Bike rental prediction: PDP for temperature

In [ ]:
bike_pdp_temp <- make_1d_pdp(
  bike_rf, bike_train, "temp", "bike count", "Normalized temperature"
)
save_plot(bike_pdp_temp, "bike_pdp_temp.png")
bike_pdp_temp

**Interpretation.** Temperature has a strong non-linear effect. Predictions are low at low temperatures, increase strongly up to warm conditions, and decrease slightly at the highest temperatures. Extreme temperature values are sparser, so they should be interpreted more cautiously.

## 7. Bike rental prediction: PDP for humidity

In [ ]:
bike_pdp_hum <- make_1d_pdp(
  bike_rf, bike_train, "hum", "bike count", "Normalized humidity"
)
save_plot(bike_pdp_hum, "bike_pdp_hum.png")
bike_pdp_hum

**Interpretation.** Humidity is almost flat to slightly positive at low and medium values, but high humidity produces a strong decrease in predicted rentals. The far-right region is less dense and should be treated with caution.

## 8. Bike rental prediction: PDP for wind speed

In [ ]:
bike_pdp_windspeed <- make_1d_pdp(
  bike_rf, bike_train, "windspeed", "bike count", "Normalized wind speed"
)
save_plot(bike_pdp_windspeed, "bike_pdp_windspeed.png")
bike_pdp_windspeed

**Interpretation.** Wind speed has a mostly decreasing effect. The Random Forest predicts fewer rentals as wind speed increases, especially after medium wind values. Very high wind speeds are sparse, so the exact shape there is less reliable.

## 9. Bike rental prediction: 2D PDP for temperature and humidity

The two-dimensional PDP shows the joint model effect of temperature and humidity. A sample is used so the computation remains efficient in Colab.

In [ ]:
bike_pdp_sample <- bike_train %>%
  sample_n(size = min(500, nrow(bike_train)))

bike_pdp_temp_hum <- partial(
  object = bike_rf,
  pred.var = c("temp", "hum"),
  train = bike_pdp_sample,
  grid.resolution = 35
)

bike_pdp_temp_hum_2d <- ggplot(bike_pdp_temp_hum, aes(x = temp, y = hum, fill = yhat)) +
  geom_tile() +
  scale_fill_viridis_c(option = "C", name = "Predicted cnt") +
  labs(
    x = "Normalized temperature",
    y = "Normalized humidity",
    title = "Bike rental model: 2D PDP for temperature and humidity"
  )

save_plot(bike_pdp_temp_hum_2d, "bike_pdp_temp_hum_2d.png", width = 7, height = 5.5)
bike_pdp_temp_hum_2d

**Interpretation.** The highest predicted counts occur at medium-high temperatures and low or moderate humidity. Cold temperatures lead to lower predictions, and high humidity reduces predictions even when temperature is warm.

## 10. Bike rental prediction: distribution of temperature and humidity

The distribution plot is necessary because PDPs are less reliable in regions with few observations.

In [ ]:
bike_temp_hum_distribution <- ggplot(bike_train, aes(x = temp, y = hum)) +
  geom_point(alpha = 0.45, color = "#2b2b2b", size = 1.6) +
  geom_density_2d(color = "#d95f02", linewidth = 0.7) +
  labs(
    x = "Normalized temperature",
    y = "Normalized humidity",
    title = "Observed distribution of temperature and humidity"
  )

save_plot(bike_temp_hum_distribution, "bike_temp_hum_distribution.png", width = 7, height = 5.5)
bike_temp_hum_distribution

**Interpretation.** The observed data are concentrated around moderate humidity and medium to high temperatures. PDP regions with few points should be interpreted carefully because the model is averaging predictions over feature combinations with limited empirical support.

**Answer to the bike exercise question.** Predicted bike rentals increase over much of the observed timeline, increase with temperature until warm values, decrease at high humidity, and decrease as wind speed grows. The 2D PDP shows that warm and relatively dry conditions produce the highest predicted rental counts.

## 11. House price prediction: data preparation and model training

The house model predicts `price` using the requested structural variables. An 80/20 train/test split is again used before interpretation.

In [ ]:
house_data <- house_raw %>%
  select(price, bedrooms, bathrooms, sqft_living, sqft_lot, floors, yr_built) %>%
  mutate(across(everything(), as.numeric)) %>%
  na.omit()

house_split <- train_test_split(house_data)
house_train <- house_split$train
house_test <- house_split$test

house_rf <- randomForest(
  price ~ bedrooms + bathrooms + sqft_living + sqft_lot + floors + yr_built,
  data = house_train,
  ntree = 200,
  nodesize = 10,
  importance = TRUE
)

house_predictions <- predict(house_rf, newdata = house_test)
house_metrics <- regression_metrics(house_test$price, house_predictions) %>%
  mutate(Dataset = "House prices", .before = RMSE)

house_rf
house_metrics

## 12. Model performance summary

PDP interpretation is more meaningful when the model has acceptable predictive quality. The table below summarises the test performance of both models.

In [ ]:
performance_table <- bind_rows(bike_metrics, house_metrics) %>%
  mutate(
    RMSE = round(RMSE, 2),
    MAE = round(MAE, 2),
    R2 = round(R2, 3)
  )

performance_table

## 13. House price prediction: PDP for bedrooms

In [ ]:
house_pdp_sample <- house_train %>%
  sample_n(size = min(1500, nrow(house_train)))

house_pdp_bedrooms <- make_1d_pdp(
  house_rf, house_pdp_sample, "bedrooms", "price", "Bedrooms",
  grid.resolution = 20
)
save_plot(house_pdp_bedrooms, "house_pdp_bedrooms.png")
house_pdp_bedrooms

**Interpretation.** Bedrooms have a non-linear effect and do not show a simple positive relationship. The model prediction decreases after the first values, which means that, once the other selected variables are averaged over, more bedrooms are not used as a direct positive price signal. Sparse high-bedroom regions require caution.

## 14. House price prediction: PDP for bathrooms

In [ ]:
house_pdp_bathrooms <- make_1d_pdp(
  house_rf, house_pdp_sample, "bathrooms", "price", "Bathrooms",
  grid.resolution = 20
)
save_plot(house_pdp_bathrooms, "house_pdp_bathrooms.png")
house_pdp_bathrooms

**Interpretation.** Bathrooms have a clear increasing effect. The Random Forest predicts higher prices as the number of bathrooms grows, especially in the medium-high range. The highest bathroom counts are less common, so the exact slope there is less reliable.

## 15. House price prediction: PDP for living area

In [ ]:
house_pdp_sqft_living <- make_1d_pdp(
  house_rf, house_pdp_sample, "sqft_living", "price", "Living area (sqft)",
  grid.resolution = 30
)
save_plot(house_pdp_sqft_living, "house_pdp_sqft_living.png")
house_pdp_sqft_living

**Interpretation.** Living area has the strongest positive effect. Predicted price increases steadily as `sqft_living` grows. Very large houses are less frequent, so the largest values should be interpreted more cautiously.

## 16. House price prediction: PDP for floors

In [ ]:
house_pdp_floors <- make_1d_pdp(
  house_rf, house_pdp_sample, "floors", "price", "Floors",
  grid.resolution = 20
)
save_plot(house_pdp_floors, "house_pdp_floors.png")
house_pdp_floors

**Interpretation.** Floors have a positive step-like effect because the variable takes a limited set of values. The model predicts higher prices for houses with more floors, although the effect is smaller than the effect of living area.

**Answer to the house exercise question.** Bedrooms show a non-linear pattern and are not simply increasing. Bathrooms increase predicted price, living area has the strongest positive effect, and floors increase prediction in a smaller step-like way.

## 17. PDP limitations and final conclusions

PDP is a global method: it shows average behaviour of the fitted model, not individual behaviour. PDPs can be misleading when the analysed features are strongly correlated with other variables, or when the plot includes sparse regions with few observations. PDPs explain the Random Forest model and should not be interpreted as causal evidence.

Overall, the bike model predicts highest demand under warm and relatively dry conditions, with lower demand for high humidity and wind. The house model relies strongly on living area, with additional positive effects from bathrooms and floors.